# MLP Optimisation Comparison

Compare the **Random Search** and **Genetic Algorithm** results produced from the uploaded MLP notebook. Selection is based only on validation F1; the test set is not accessed here.

In [ ]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / 'mlp'
OPTIMISATION_DIR = OUTPUT_DIR / "optimisation"
random_path = OPTIMISATION_DIR / "random_search_best.json"
ga_path = OPTIMISATION_DIR / "genetic_algorithm_best.json"
for path in [random_path, ga_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run both optimisation notebooks first.")
print("Optimisation directory:", OPTIMISATION_DIR)

In [ ]:
with open(random_path, "r", encoding="utf-8") as f:
    random_best = json.load(f)
with open(ga_path, "r", encoding="utf-8") as f:
    ga_best = json.load(f)

comparison_df = pd.DataFrame([
    {"method": random_best["method"], "validation_f1": random_best["validation_f1"], "validation_accuracy": random_best["validation_accuracy"], "best_val_loss": random_best["best_val_loss"], "config": json.dumps(random_best["config"], sort_keys=True)},
    {"method": ga_best["method"], "validation_f1": ga_best["validation_f1"], "validation_accuracy": ga_best["validation_accuracy"], "best_val_loss": ga_best["best_val_loss"], "config": json.dumps(ga_best["config"], sort_keys=True)},
]).sort_values(["validation_f1", "best_val_loss"], ascending=[False, True])
display(comparison_df)

selected_source = random_best if (random_best["validation_f1"], -random_best["best_val_loss"]) >= (ga_best["validation_f1"], -ga_best["best_val_loss"]) else ga_best
selected = {"method": selected_source["method"], "validation_f1": selected_source["validation_f1"], "validation_accuracy": selected_source["validation_accuracy"], "best_val_loss": selected_source["best_val_loss"], "config": selected_source["config"]}
selected_path = OPTIMISATION_DIR / "selected_hyperparameters.json"
comparison_path = OPTIMISATION_DIR / "optimisation_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
with open(selected_path, "w", encoding="utf-8") as f:
    json.dump(selected, f, indent=2)
print("Selected method:", selected["method"])
print("Selected hyperparameters:", selected["config"])
print("Saved:", selected_path)

## Next step

Run the **Final Selected Model** notebook. That notebook loads `selected_hyperparameters.json`, retrains the selected configuration using the original workflow, and accesses the held-out test set once.